# 02 Inventory Clean

This notebook cleans the inventory workbook.

Important rule:
- Do not infer or fill missing values.
- Do not back-calculate missing fields.
- Keep reported values as reported.

The input file must be in the same folder as this notebook.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('.').resolve()
OUTPUT_DIR = BASE_DIR / 'inventory_clean_output'
OUTPUT_DIR.mkdir(exist_ok=True)

inventory_file_name = input(
    'Enter the inventory workbook file name in this folder: '
).strip()

if not inventory_file_name:
    raise ValueError('You must enter an inventory workbook file name.')

INVENTORY_FILE_PATH = BASE_DIR / inventory_file_name

if not INVENTORY_FILE_PATH.exists():
    raise FileNotFoundError(f'File not found: {INVENTORY_FILE_PATH}')

print('Inventory file:', INVENTORY_FILE_PATH)
print('Output folder:', OUTPUT_DIR)

In [ ]:
INVENTORY_COLUMNS = [
    'pwsid',
    'supply_name',
    'lead_lines',
    'gpcl_lines',
    'non_lead_lines',
    'unknown_lines',
    'total_lines',
]

NUMERIC_COLUMNS = [
    'lead_lines',
    'gpcl_lines',
    'non_lead_lines',
    'unknown_lines',
    'total_lines',
]


def load_inventory_workbook(path):
    df = pd.read_excel(path, header=2)
    df = df.iloc[:, 0:7].copy()
    df.columns = INVENTORY_COLUMNS

    df['pwsid'] = df['pwsid'].astype(str).str.strip()
    df['supply_name'] = df['supply_name'].astype(str).str.strip()

    for col in NUMERIC_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    raw_rows = len(df)

    df = df.dropna(subset=['pwsid']).copy()

    exact_duplicates = df[df.duplicated(keep=False)].sort_values(['pwsid', 'supply_name']).reset_index(drop=True)
    pwsid_duplicates = df[
        df.duplicated(subset=['pwsid'], keep=False)
    ].sort_values(['pwsid', 'supply_name']).reset_index(drop=True)

    clean_inventory = df.drop_duplicates().copy()

    # No filling or inference: only calculate QA fields when the reported components exist.
    clean_inventory['calc_total_from_reported_components'] = clean_inventory[
        ['lead_lines', 'gpcl_lines', 'non_lead_lines', 'unknown_lines']
    ].sum(axis=1, min_count=4)

    clean_inventory['inventory_missing_any'] = clean_inventory[NUMERIC_COLUMNS].isna().any(axis=1)

    clean_inventory['total_mismatch'] = pd.NA
    comparable_mask = (
        clean_inventory['calc_total_from_reported_components'].notna()
        & clean_inventory['total_lines'].notna()
    )
    clean_inventory.loc[comparable_mask, 'total_mismatch'] = (
        clean_inventory.loc[comparable_mask, 'calc_total_from_reported_components']
        != clean_inventory.loc[comparable_mask, 'total_lines']
    )

    clean_inventory['total_to_identify_or_replace'] = clean_inventory[
        ['lead_lines', 'gpcl_lines', 'unknown_lines']
    ].sum(axis=1, min_count=3)

    summary = pd.DataFrame([
        {
            'raw_rows': raw_rows,
            'rows_after_drop_missing_pwsid': len(df),
            'exact_duplicate_rows': len(exact_duplicates),
            'duplicate_pwsid_rows': len(pwsid_duplicates),
            'clean_inventory_rows': len(clean_inventory),
            'inventory_missing_any_rows': int(clean_inventory['inventory_missing_any'].sum()),
            'total_mismatch_true_rows': int((clean_inventory['total_mismatch'] == True).sum()),
        }
    ])

    clean_inventory = clean_inventory.sort_values(['pwsid', 'supply_name']).reset_index(drop=True)

    return clean_inventory, exact_duplicates, pwsid_duplicates, summary

In [ ]:
clean_inventory, exact_duplicates, pwsid_duplicates, summary_df = load_inventory_workbook(INVENTORY_FILE_PATH)

display(summary_df)
display(clean_inventory.head())
display(pwsid_duplicates.head())

In [ ]:
output_stem = Path(inventory_file_name).stem

clean_path = OUTPUT_DIR / f'{output_stem}_clean.csv'
exact_dup_path = OUTPUT_DIR / f'{output_stem}_exact_duplicates.csv'
pwsid_dup_path = OUTPUT_DIR / f'{output_stem}_duplicate_pwsid_rows.csv'

clean_inventory.to_csv(clean_path, index=False)
exact_duplicates.to_csv(exact_dup_path, index=False)
pwsid_duplicates.to_csv(pwsid_dup_path, index=False)

print('Saved:', clean_path)
print('Saved:', exact_dup_path)
print('Saved:', pwsid_dup_path)